In [ ]:
import gzip
import h5py
import numpy as np
import matplotlib
#matplotlib.use('Agg')
%matplotlib inline  
matplotlib.rcParams.update({'font.size': 12})
import matplotlib.pyplot as plt

DIM=3

def loadmeta(dir="./ckpt", rank=0):
    fmeta=f"{dir}/ckpt.meta.{rank}"
    nx = np.zeros([3],dtype=np.int32)
    bbox=np.zeros([DIM,2])

    f = open(fmeta, 'r')
    f.readline()  ## skip the first line
    dt, nx[0],nx[1],nx[2], nv, nphi = f.readline().split()
    for d in range(DIM):
        bbox[d,0], bbox[d,1]= f.readline().split()
        f.readline()
    nv,nphi=int(nv),int(nphi)    
    vx=np.array(f.readline().split(),dtype=np.double).reshape([nv,nphi])
    vy=np.array(f.readline().split(),dtype=np.double).reshape([nv,nphi])
    vz=np.array(f.readline().split(),dtype=np.double).reshape([nv,nphi])
    vw=np.array(f.readline().split(),dtype=np.double).reshape([nv,nphi])

    #z_grid = [ float(x.strip()) for x in f.readline().split() ]
    #v_grid = [ float(x.strip()) for x in
    #f.readline().split() ]
    ##zgrid = np.fromfile(fn, dtype=np.double, count=sz)
    ##vgrid = np.fromfile(fn, dtype=np.double, count=sv)
    #dz = (float(z1)-float(z0))/int(nz)

    return nx, int(nv), int(nphi), bbox, vx,vy,vz,vw

def load(iter=0, dir="./ckpt", f=0,rank=0):
    nx, nv, nphi, bbox, _,_,_,_ = loadmeta(dir,rank)
    FNAME=f"{dir}/it{iter}/ckpt{f}.{rank}"
    with gzip.open(FNAME, 'rb') as f:
        fread = f.read()
        iter = np.frombuffer(fread, dtype=np.int32,  count=1, offset=0)[0]
        time = np.frombuffer(fread, dtype=np.double, count=1, offset=4)[0]
        arr  = np.frombuffer(fread, dtype=np.double, count=nx[0]*nx[1]*nx[2]*nv*nphi, offset=12).reshape([nx[0],nx[1],nx[2],nv, nphi])
    print(f"Load file {FNAME} at Iter={iter}  T={time}")
    return arr  

In [ ]:
FOLDER="../ben_fd8/b100z/v064p02/ckpt"
FOLDER="../ben_fd8/b150/v064p02/ckpt"
FOLDER="../../nuosc3d_test/bench/fz7/v032p32/ckpt"
FOLDER="../../nuosc3d_xyz/fluxXY/ben_fd8/fXY45_6/v016p32/ckpt"
FOLDER="../../nuosc3d_xyz/fluxX/ben_fd8/fZ6/v016p32/ckpt"
FOLDER="../../nuosc3d_xyz/fluxX/ben_fd8/fX6/v016p32/ckpt"
FOLDER="../../nuosc3d_xyz_icosa/fluxXY/ben_fd8/testXY/v008p99/ckpt"

nx, nv, nphi, bbox, vx,vy,vz,vw = loadmeta(FOLDER)
print (nx, nv, nphi)
print (bbox)
usize = nx[0]*nx[1]*nx[2]*nv*nphi
print(usize*8*2)
s0,s1=bbox[0,:]
l0,l1=bbox[1,:]
z0,z1=bbox[2,:]

In [ ]:
FNAME=f"{FOLDER}/eln.{0}"
with gzip.open(FNAME, 'rb') as f:
 fread = f.read()
 g0  = np.frombuffer(fread, dtype=np.double, count=usize, offset=0      ).reshape([nx[0],nx[1],nx[2],nv,nphi])
 g0b = np.frombuffer(fread, dtype=np.double, count=usize, offset=8*usize).reshape([nx[0],nx[1],nx[2],nv,nphi])

In [ ]:
f=1
iter = 2000
FX="../../nuosc3d_xyz/fluxX/ben_fd8/fX6/v016p32/ckpt"
FY="../../nuosc3d_xyz/fluxX/ben_fd8/fY6/v016p32/ckpt"
FZ="../../nuosc3d_xyz/fluxX/ben_fd8/fZ6/v016p32/ckpt"
nx, nv, nphi, bbox, vw = loadmeta(FX)
l0,l1=bbox[0,:]
s0,s1=bbox[1,:]
arrX=load(iter, dir=FX, f=f)
arrY=load(iter, dir=FY, f=f)
arrZ=load(iter, dir=FZ, f=f)
pX= np.sum(np.sum(arrX, axis=-1), axis=-1)
pY= np.sum(np.sum(arrY, axis=-1), axis=-1)
pZ= np.sum(np.sum(arrZ, axis=-1), axis=-1)
print(pX.shape, pY.shape, pZ.shape)

In [ ]:
plt.figure(figsize=(10, 2))
im = plt.imshow(pX[:,:,0].T, extent=[l0, l1, s0, s1], origin='lower', aspect='auto', interpolation='none')
plt.colorbar(im)
plt.xlabel(f"X: iter={iter}")
plt.ylabel("Y")
plt.show()

plt.figure(figsize=(10, 2))
im = plt.imshow(pY[0,:,:].T, extent=[l0, l1, s0, s1], origin='lower', aspect='auto', interpolation='none')
plt.colorbar(im)
plt.xlabel(f"Y: iter={iter}")
plt.ylabel("Z")
plt.show()

plt.figure(figsize=(10, 2))
im = plt.imshow(pZ[:,0,:], extent=[l0, l1, -1, 1], origin='lower', aspect='auto', interpolation='none')
plt.colorbar(im)
plt.xlabel(f"Z: iter={iter}")
plt.ylabel("X")
plt.show()


In [ ]:
f=1
iter = 0
FX="../../nuosc3d_xyz/fluxX/ben_fd8/fX6/v016p32/ckpt"
FY="../../nuosc3d_xyz/fluxX/ben_fd8/fY6/v016p32/ckpt"
FZ="../../nuosc3d_xyz/fluxX/ben_fd8/fZ6/v016p32/ckpt"
nx, nv, nphi, bbox = loadmeta(FX)
l0,l1=bbox[0,:]
s0,s1=bbox[1,:]
arrX = load(iter, dir=FX, f=f)
arrY = load(iter, dir=FY, f=f)
arrZ = load(iter, dir=FZ, f=f)

plt.figure(figsize=(10, 2))
im = plt.imshow(arrX[:,0,0,0,0].T, extent=[l0, l1, -1, 1], origin='lower', aspect='auto', interpolation='none')
plt.colorbar(im)
plt.xlabel(f"X: iter={iter}")
plt.ylabel("V")
plt.show()

plt.figure(figsize=(10, 2))
im = plt.imshow(arrY[0,:,0,:,0].T, extent=[l0, l1, -1, 1], origin='lower', aspect='auto', interpolation='none')
plt.colorbar(im)
plt.xlabel(f"Y: iter={iter}")
plt.ylabel("V")
plt.show()

plt.figure(figsize=(10, 2))
im = plt.imshow(arr[0,0,:,:,0].T, extent=[l0, l1, -1, 1], origin='lower', aspect='auto', interpolation='none')
plt.colorbar(im)
plt.xlabel(f"Z: iter={iter}")
plt.ylabel("V")
plt.show()


In [ ]:
#### Plot Y
FOLDER="../../nuosc3d_xyz/fluxX/ben_fd8/fY6/v016p32/ckpt"
FOLDER="../../nuosc3d_xyz/fluxX/ben_fd8/fY6v/v016p32/ckpt"
nx, nv, nphi, bbox, _ = loadmeta(FOLDER)
x0,x1=bbox[0,:]
y0,y1=bbox[1,:]
z0,z1=bbox[2,:]
f=0
for iter in range(0,200,40):
  arr = load(iter, dir=FOLDER, f=f)
  plt.figure(figsize=(10, 2))
  im = plt.imshow(arr[1,:,1,:,10].T, extent=[y0, y1, -1, 1], origin='lower', aspect='auto', interpolation='none')
  plt.colorbar(im)
  plt.xlabel(f"Y: iter={iter}")
  plt.ylabel("V")
  plt.show()


In [ ]:
#### Plot XY
nx, nv, nphi, bbox, vx,vy,vz,vw = loadmeta(FOLDER)
x0,x1=bbox[0,:]
y0,y1=bbox[1,:]
z0,z1=bbox[2,:]
f=0
iv,ip=0,11
for iter in range(0,200,20):
  arr = load(iter, dir=FOLDER, f=f)
  plt.figure(figsize=(5, 5))
  im = plt.imshow(arr[:,:,1,iv,ip].T, extent=[y0, y1, x0, y1], origin='lower', aspect='auto', interpolation='none')
  plt.colorbar(im)
  plt.xlabel(f"Y: iter={iter}  v=({vx[iv,ip]} {vy[iv,ip]} {vz[iv,ip]})")
  plt.ylabel("X")
  plt.show()


In [ ]:
#### Plot Z
FOLDER="../../nuosc3d_xyz/fluxX/ben_fd8/fZ6/v016p32/ckpt"
nx, nv, nphi, bbox, _ = loadmeta(FOLDER)
x0,x1=bbox[0,:]
y0,y1=bbox[1,:]
z0,z1=bbox[2,:]
f=0
for iter in range(0,2001,400):
  arr = load(iter, dir=FOLDER, f=f)
  plt.figure(figsize=(10, 2))
  im = plt.imshow(arr[0,1,:,:,20].T, extent=[z0, z1, -1, 1], origin='lower', aspect='auto', interpolation='none')
  plt.colorbar(im)
  plt.xlabel(f"Y: iter={iter}")
  plt.ylabel("V")
  plt.show()


In [ ]:
f=0
for iter in range(0,2001,200):
  arr = load(iter, dir=FOLDER, f=f)
  plt.figure(figsize=(10, 2))
  im = plt.imshow(arr[0,0,:,:,0].T, extent=[z0, z1, -1, 1], origin='lower', aspect='auto', interpolation='none')
  plt.colorbar(im)
  plt.xlabel(f"Z: iter={iter}")
  plt.ylabel("V")
  plt.show()


In [ ]:
iter=0
iv=3
for f in range(8):
  arr = load(iter, dir=FOLDER, f=f)
  plt.figure(figsize=(12, 6))
  y0,y1=bbox[1,:]
  z0,z1=bbox[2,:]
  im = plt.imshow(arr[0,:,:,iv], extent=[z0, z1, y0, y1], origin='lower', aspect='auto', interpolation='none')
  plt.colorbar(im)
  plt.xlabel("Z")
  plt.ylabel("Y")
  plt.show()


In [ ]:
iter=0
iv=3
iz=22
for f in range(8):
  arr = load(iter, dir=FOLDER, f=f)
  plt.figure(figsize=(5, 4))
  y0,y1=bbox[1,:]
  x0,x1=bbox[0,:]
  im = plt.imshow(arr[:,:,iz,iv], extent=[x0, x1, y0, y1], origin='lower', aspect=1, interpolation='none')
  plt.colorbar(im)
  plt.xlabel("X")
  plt.ylabel("Y")
  plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def fibonacci_sphere(samples=1):
    points = []
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle in radians

    for i in range(samples):
        y = 1 - (i / float(samples - 1)) * 2  # y goes from 1 to -1
        radius = np.sqrt(1 - y * y)  # radius at y

        theta = phi * i  # golden angle increment

        x = np.cos(theta) * radius
        z = np.sin(theta) * radius

        points.append([x, y, z])

    return np.array(points)

# Number of points
num_points = 2000

# Generate uniform points on a sphere
points = fibonacci_sphere(num_points)

# Plot the points
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=1, c='b', alpha=0.5)

# Set equal aspect ratio for all axes
ax.set_box_aspect([1,1,1])

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Uniform Points on a Unit Sphere')

plt.show()